# CRISP-DM Phase 3: Data Preparation (Sirio-Libanes Clustering)

In this phase, we prepare our dataset for unsupervised grouping! Our primary battles here are:
1. **50% Missing Data Bomb**: We must calculate values based on patient longitudinal history.
2. **The Curse of Dimensionality**: We have 231 columns. Calculating mathematical clusters in a 231-axis geometric space is nearly impossible. We will deploy **PCA** to mathematically crush this dataset into heavy super-columns while preserving 95% of the truth!

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
We pull the raw 231-column dataset directly from the Kaggle Excel file.

In [2]:
DATA_FILE = 'Kaggle_Sirio_Libanes_ICU_Prediction.xlsx'
df = pd.read_excel(DATA_FILE)
print(f"Raw Data Loaded. Dimensions: {df.shape}")

Raw Data Loaded. Dimensions: (1925, 231)


## 2. Longitudinal Patient-Aware Imputation
**Why are we doing this?** If Patient A misses a blood pressure reading at hour 4, we shouldn't use the global average of other patients! Instead, we group the data by `PATIENT_VISIT_IDENTIFIER`. We use **Forward-Fill** (take their reading from hour 2 and push it forward) and **Backward-Fill** (take their reading from hour 6 and push it back) so the data is completely personalized to their timeline!

In [3]:
# 1. Group by patient and apply forward/backward filling natively
# We use group_keys=False to avoid multi-index duplication in recent pandas versions
df_grouped = df.groupby('PATIENT_VISIT_IDENTIFIER', group_keys=False).apply(lambda x: x.ffill().bfill())

# 2. If a patient missed a test entirely for all 5 windows, the ffill fails. 
# We safely catch the remaining NaNs using the global statistical median.
# We must explicitly select numeric columns for median calculation to avoid errors.
num_cols = df_grouped.select_dtypes(include=[np.number]).columns
df_grouped[num_cols] = df_grouped[num_cols].fillna(df_grouped[num_cols].median())

print(f"Global Missing Rate after our Custom Imputation: {df_grouped.isna().mean().sum():.2f}%")

Global Missing Rate after our Custom Imputation: 0.00%


## 3. Ordinal Mathematical Encoding
`AGE_PERCENTIL` is stored as text ("10th", "20th"). Clustering algorithms measure spatial distance. Text has zero distance! We manually map these text blocks into hard ordinal tiers (1 through 10).

In [4]:
age_map = {
    '10th': 1, '20th': 2, '30th': 3, '40th': 4, '50th': 5,
    '60th': 6, '70th': 7, '80th': 8, '90th': 9, 'Above 90th': 10
}
df_grouped['AGE_NUMERIC'] = df_grouped['AGE_PERCENTIL'].map(age_map)

# Drop the old text column, our Patient ID, Window, and the Target ICU column!
# Why drop ICU? Because this is unsupervised clustering! The algorithm must find groupings on its own without knowing who went to the ICU.
X = df_grouped.drop(['AGE_PERCENTIL', 'PATIENT_VISIT_IDENTIFIER', 'WINDOW', 'ICU'], axis=1)
print("Structural columns removed to form pure clustering matrix.")

Structural columns removed to form pure clustering matrix.


## 4. Standard Scaling
Since the algorithms will be plotting 230-dimensional graphs, large numbers (Heart Rate) will look way bigger than small decimals (pH). `StandardScaler` perfectly flattens all metrics onto the exact same playing field so algorithms can detect the hidden patterns fairly.

In [5]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print("Matrix Scaled successfully.")

Matrix Scaled successfully.


## 5. Crushing Dimensionality using PCA
The Curse of Dimensionality states that if you try to group things across 230 axes, everything is magically the same distance apart mathematically. 

**Principal Component Analysis (PCA)** is an advanced mathematical transformation that takes our 230 columns and crushes them down while strictly preserving **95%** of the entire dataset's variance and behavior. Let's see how many columns we really need to keep 95% of the information!

In [6]:
pca = PCA(n_components=0.95, random_state=42)
pca_matrix = pca.fit_transform(X_scaled)

X_pca = pd.DataFrame(pca_matrix, columns=[f"PC_{i+1}" for i in range(pca_matrix.shape[1])])

print(f"Original Columns: {X_scaled.shape[1]}")
print(f"PCA Crushed Columns: {X_pca.shape[1]}")
print(f"We successfully preserved 95% of the information while destroying {(1 - (X_pca.shape[1]/X_scaled.shape[1]))*100:.1f}% of the unnecessary columns!")

Original Columns: 228
PCA Crushed Columns: 40
We successfully preserved 95% of the information while destroying 82.5% of the unnecessary columns!


## 6. Export Final Matrices
We export the raw scaled version and the PCA version just in case we need to benchmark them.

In [7]:
out_dir = os.path.join('data', 'processed', 'sirio')
os.makedirs(out_dir, exist_ok=True)

X_scaled.to_csv(os.path.join(out_dir, 'X_scaled.csv'), index=False)
X_pca.to_csv(os.path.join(out_dir, 'X_pca.csv'), index=False)

print(f"Success! Exported Final Matrices to: {os.path.abspath(out_dir)}")

Success! Exported Final Matrices to: C:\Users\rahma\Desktop\machine learning project\clustering_sirio_covid_data_set\data\processed\sirio
